# 01 — Data Collection

In [29]:
# imports
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer


In [30]:
path = 'C:/users/brady/data_science/NamUs_Missing/data/raw/'
raw_missing_data = pd.read_csv(path+'missing_persons.csv')
pd.set_option("display.max_columns", None)

In [31]:
# looking at the data to see what we are working with
raw_missing_data.head(10)

,Case Number,DLC,Legal Last Name,Legal First Name,Missing Age,City,County,State,Biological Sex,Race / Ethnicity,Date Modified
0,MP160223,07/06/2026,ARRINGTON,JACOB,14 Years,Bowling Green,Polk,FL,Male,White / Caucasian,07/07/2026
1,MP160195,07/05/2026,Motta,Gabriel,14 Years,Winter Garden,Orange,FL,Male,"White / Caucasian, Hispanic / Latino",07/06/2026
2,MP160193,07/05/2026,Julio,Star,15 Years,North Miami Beach,Miami-Dade,FL,Female,"White / Caucasian, Hispanic / Latino",07/06/2026
3,MP160196,07/05/2026,JACKSON,JEREMIAH,16 Years,Melbourne,Brevard,FL,Male,Black / African American,07/07/2026
4,MP160184,07/04/2026,ROBLES,LORENZO,17 Years,Cocoa,Brevard,FL,Male,Black / African American,07/07/2026
5,MP160185,07/04/2026,BERNARDO,STARR,17 Years,Bartow,Polk,FL,Female,Black / African American,07/07/2026
6,MP160186,07/04/2026,BERNARDO,SCYEE,16 Years,Bartow,Polk,FL,Female,Black / African American,07/07/2026
7,MP160183,07/04/2026,JENKINS,RONALD,82 Years,Mulberry,Polk,FL,Male,White / Caucasian,07/07/2026
8,MP160187,07/04/2026,PERRY,ELIZABETH,15 Years,Fort Lauderdale,Broward,FL,Female,Black / African American,07/07/2026
9,MP160180,07/03/2026,QUINN,MARK,17 Years,Rockledge,Brevard,FL,Male,White / Caucasian,07/07/2026


In [32]:
raw_missing_data.info()
raw_missing_data.shape

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Case Number       10000 non-null  str  
 1   DLC               10000 non-null  str  
 2   Legal Last Name   9999 non-null   str  
 3   Legal First Name  10000 non-null  str  
 4   Missing Age       10000 non-null  str  
 5   City              9992 non-null   str  
 6   County            9986 non-null   str  
 7   State             10000 non-null  str  
 8   Biological Sex    10000 non-null  str  
 9   Race / Ethnicity  10000 non-null  str  
 10  Date Modified     10000 non-null  str  
dtypes: str(11)
memory usage: 1.7 MB


(10000, 11)

In [33]:
#change column names to lower case and include underscores instead of spaces for consistency
raw_missing_data.columns = (
    raw_missing_data.columns
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('/', '')
    .str.replace('__', '_')
)

In [34]:
# delete rows with missing values in the county and city columns
raw_missing_data = raw_missing_data.dropna(subset=['county', 'city']).reset_index(drop=True)

In [35]:
#impute single missing last name with 'Unknown'
imputer = SimpleImputer(missing_values=np.nan, strategy='constant', fill_value='Unknown')
raw_missing_data['legal_last_name'] = imputer.fit_transform(raw_missing_data[['legal_last_name']]).ravel()

In [36]:
#check info and shape after dropping rows and imputing values to confirm changes
raw_missing_data.info()
raw_missing_data.shape

<class 'pandas.DataFrame'>
Index: 9978 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   case_number       9978 non-null   str  
 1   dlc               9978 non-null   str  
 2   legal_last_name   9978 non-null   str  
 3   legal_first_name  9978 non-null   str  
 4   missing_age       9978 non-null   str  
 5   city              9978 non-null   str  
 6   county            9978 non-null   str  
 7   state             9978 non-null   str  
 8   biological_sex    9978 non-null   str  
 9   race_/_ethnicity  9978 non-null   str  
 10  date_modified     9978 non-null   str  
dtypes: str(11)
memory usage: 1.8 MB


(9978, 11)

In [43]:
raw_missing_data.head(5)

,case_number,dlc,legal_last_name,legal_first_name,missing_age,city,county,state,biological_sex,race_/_ethnicity,date_modified
0,MP160223,2026-07-06,ARRINGTON,JACOB,14 Years,Bowling Green,Polk,FL,Male,White / Caucasian,07/07/2026
1,MP160195,2026-07-05,Motta,Gabriel,14 Years,Winter Garden,Orange,FL,Male,"White / Caucasian, Hispanic / Latino",07/06/2026
2,MP160193,2026-07-05,Julio,Star,15 Years,North Miami Beach,Miami-Dade,FL,Female,"White / Caucasian, Hispanic / Latino",07/06/2026
3,MP160196,2026-07-05,JACKSON,JEREMIAH,16 Years,Melbourne,Brevard,FL,Male,Black / African American,07/07/2026
4,MP160184,2026-07-04,ROBLES,LORENZO,17 Years,Cocoa,Brevard,FL,Male,Black / African American,07/07/2026


In [44]:
#delete the word 'years' from the age column and cast to integer
raw_missing_data['missing_age'] = raw_missing_data['missing_age'].str.replace('Years', '').str.strip()

In [ ]:
# cast rows to correct data types
# cast the 'dlc' and 'date_modified' columns to datetime, coercing errors to NaT for invalid dates
raw_missing_data['dlc'] = pd.to_datetime(raw_missing_data['dlc'], errors='coerce')
raw_missing_data['date_modified'] = pd.to_datetime(raw_missing_data['date_modified'], errors='coerce')

# clean missing_age column by converting to numeric and taking the average of any age ranges (i.e. 30-40 years would be 35)
raw_missing_data['missing_age'] = (
    raw_missing_data['missing_age'].str.replace('Years', '', regex=False).str.replace('Year', '', regex=False).str.strip()
)

# if it's a range, average the two numbers and round down; '< 1' -> 0
def fix_age(x):
    x = str(x).strip()
    if '<' in x:
        return 0
    if '-' in x:
        a, b = x.split('-')
        return (int(a) + int(b)) // 2
    return int(x)

raw_missing_data['missing_age'] = raw_missing_data['missing_age'].apply(fix_age)


In [50]:
raw_missing_data.head(5)

,case_number,dlc,legal_last_name,legal_first_name,missing_age,city,county,state,biological_sex,race_/_ethnicity,date_modified
0,MP160223,2026-07-06,ARRINGTON,JACOB,14,Bowling Green,Polk,FL,Male,White / Caucasian,2026-07-07
1,MP160195,2026-07-05,Motta,Gabriel,14,Winter Garden,Orange,FL,Male,"White / Caucasian, Hispanic / Latino",2026-07-06
2,MP160193,2026-07-05,Julio,Star,15,North Miami Beach,Miami-Dade,FL,Female,"White / Caucasian, Hispanic / Latino",2026-07-06
3,MP160196,2026-07-05,JACKSON,JEREMIAH,16,Melbourne,Brevard,FL,Male,Black / African American,2026-07-07
4,MP160184,2026-07-04,ROBLES,LORENZO,17,Cocoa,Brevard,FL,Male,Black / African American,2026-07-07


In [52]:
raw_missing_data.info()
raw_missing_data.shape

<class 'pandas.DataFrame'>
Index: 9978 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   case_number       9978 non-null   str           
 1   dlc               9978 non-null   datetime64[ns]
 2   legal_last_name   9978 non-null   str           
 3   legal_first_name  9978 non-null   str           
 4   missing_age       9978 non-null   Int64         
 5   city              9978 non-null   str           
 6   county            9978 non-null   str           
 7   state             9978 non-null   str           
 8   biological_sex    9978 non-null   str           
 9   race_/_ethnicity  9978 non-null   str           
 10  date_modified     9978 non-null   datetime64[us]
dtypes: Int64(1), datetime64[ns](1), datetime64[us](1), str(8)
memory usage: 1.5 MB


(9978, 11)